### Pin classification with SigLIP embeddings + MLP (img only)


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from PIL import Image
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModel, AutoProcessor
from peft import LoraConfig, TaskType, get_peft_model

BACKEND = 'img_only'
MODEL_ID = 'google/siglip-so400m-patch14-384'
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts" / "pin_classification"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
PROCESSED_CANDIDATES = [
    PROJECT_ROOT / "data" / "processed" / "places365_pin_manifest.parquet",
    PROJECT_ROOT / "data" / "processed" / "food_101_pin_manifest.parquet",
    PROJECT_ROOT / "data" / "processed" / "inaturalist_pin_manifest.parquet",
]

model = AutoModel.from_pretrained(MODEL_ID)
processor = AutoProcessor.from_pretrained(MODEL_ID)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device).eval()
sns.set_theme(style="whitegrid")


In [ ]:
def compute_multiclass_metrics(y_true, probas, classes) -> dict:
    from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
    from sklearn.preprocessing import label_binarize

    y_true = np.asarray(y_true)
    probas = np.asarray(probas)
    pred_idx = probas.argmax(axis=1)
    pred_labels = np.asarray(classes)[pred_idx]

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, pred_labels, average="macro", zero_division=0
    )

    y_bin = label_binarize(y_true, classes=classes)
    try:
        auc = roc_auc_score(y_bin, probas, average="macro", multi_class="ovr")
    except ValueError:
        auc = float("nan")

    fpr_values = []
    for class_index, class_name in enumerate(classes):
        y_pos = (y_true == class_name).astype(int)
        pred_pos = (pred_labels == class_name).astype(int)
        fp = int(((pred_pos == 1) & (y_pos == 0)).sum())
        tn = int(((pred_pos == 0) & (y_pos == 0)).sum())
        fpr_values.append(fp / max(fp + tn, 1))

    return {
        "Precision": round(float(precision), 4),
        "Recall": round(float(recall), 4),
        "F1": round(float(f1), 4),
        "FPR": round(float(np.mean(fpr_values)), 4),
        "AUC": round(float(auc), 4) if not np.isnan(auc) else None,
    }


def load_pin_frame() -> pd.DataFrame:
    for candidate in PROCESSED_CANDIDATES:
        if candidate.exists():
            frame = pd.read_parquet(candidate)
            frame["source_dataset"] = candidate.stem.replace("_pin_manifest", "")
            return frame
    raise FileNotFoundError("Run one of the pin dataset notebooks first.")


pin_df = load_pin_frame()
pin_df["image_paths"] = pin_df["image_paths_json"].map(json.loads)
pin_df["text_bundle"] = (
    pin_df["title"].fillna("") + " [SEP] " +
    pin_df["description"].fillna("") + " [SEP] " +
    pin_df["board_name"].fillna("") + " [SEP] " +
    pin_df["tags"].fillna("")
)
classes = sorted(pin_df["label_name"].unique().tolist())
class_to_idx = {label: idx for idx, label in enumerate(classes)}

train_df, valid_df = train_test_split(
    pin_df,
    test_size=0.2 if len(pin_df) >= 100 else 0.3,
    stratify=pin_df["label_name"] if pin_df["label_name"].nunique() > 1 else None,
    random_state=42,
)


In [ ]:
@torch.inference_mode()
def image_embedding(paths: list[str]) -> np.ndarray:
    images = [Image.open(path).convert("RGB") for path in paths]
    inputs = processor(images=images, return_tensors="pt").to(device)
    feats = model.get_image_features(**inputs).detach().cpu().numpy()
    return feats.mean(axis=0)


@torch.inference_mode()
def text_embedding(text: str) -> np.ndarray:
    inputs = processor(text=[text], return_tensors="pt", padding=True, truncation=True).to(device)
    return model.get_text_features(**inputs).detach().cpu().numpy()[0]


sample_image_emb = image_embedding(valid_df.iloc[0]["image_paths"])
sample_text_emb = text_embedding(valid_df.iloc[0]["text_bundle"])
plt.figure(figsize=(10, 4))
sns.heatmap(pd.DataFrame(np.vstack([sample_image_emb[:128], sample_text_emb[:128]])).T, cmap="mako")
plt.title("SigLIP embedding slices")
plt.tight_layout()


In [ ]:
def compute_multiclass_metrics(y_true, probas, classes) -> dict:
    from sklearn.metrics import precision_recall_fscore_support, roc_auc_score
    from sklearn.preprocessing import label_binarize

    y_true = np.asarray(y_true)
    probas = np.asarray(probas)
    pred_idx = probas.argmax(axis=1)
    pred_labels = np.asarray(classes)[pred_idx]

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, pred_labels, average="macro", zero_division=0
    )

    y_bin = label_binarize(y_true, classes=classes)
    try:
        auc = roc_auc_score(y_bin, probas, average="macro", multi_class="ovr")
    except ValueError:
        auc = float("nan")

    fpr_values = []
    for class_index, class_name in enumerate(classes):
        y_pos = (y_true == class_name).astype(int)
        pred_pos = (pred_labels == class_name).astype(int)
        fp = int(((pred_pos == 1) & (y_pos == 0)).sum())
        tn = int(((pred_pos == 0) & (y_pos == 0)).sum())
        fpr_values.append(fp / max(fp + tn, 1))

    return {
        "Precision": round(float(precision), 4),
        "Recall": round(float(recall), 4),
        "F1": round(float(f1), 4),
        "FPR": round(float(np.mean(fpr_values)), 4),
        "AUC": round(float(auc), 4) if not np.isnan(auc) else None,
    }


def row_to_features(row: pd.Series) -> np.ndarray:
    img = image_embedding(row["image_paths"])
    if BACKEND == "img_only":
        return img
    txt = text_embedding(row["text_bundle"])
    return np.concatenate([img, txt], axis=0)


train_x = np.vstack(train_df.apply(row_to_features, axis=1).tolist())
valid_x = np.vstack(valid_df.apply(row_to_features, axis=1).tolist())
train_y = np.array([class_to_idx[label] for label in train_df["label_name"]])
valid_y = np.array([class_to_idx[label] for label in valid_df["label_name"]])

train_ds = TensorDataset(torch.tensor(train_x, dtype=torch.float32), torch.tensor(train_y, dtype=torch.long))
valid_ds = TensorDataset(torch.tensor(valid_x, dtype=torch.float32), torch.tensor(valid_y, dtype=torch.long))
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=64)


class PinMLP(nn.Module):
    def __init__(self, in_dim: int, num_classes: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 1024),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        return self.net(x)


mlp = PinMLP(train_x.shape[1], len(classes)).to(device)
optimizer = torch.optim.AdamW(mlp.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
history = []
for epoch in range(5):
    mlp.train()
    running_loss = 0.0
    for features, labels in train_loader:
        features, labels = features.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = mlp(features)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * len(labels)

    mlp.eval()
    all_probs = []
    with torch.no_grad():
        for features, _ in valid_loader:
            probs = torch.softmax(mlp(features.to(device)), dim=-1).cpu().numpy()
            all_probs.append(probs)
    all_probs = np.vstack(all_probs)
    metrics = compute_multiclass_metrics(valid_df["label_name"], all_probs, classes)
    metrics["epoch"] = epoch + 1
    metrics["loss"] = round(running_loss / max(len(train_df), 1), 4)
    history.append(metrics)

history_df = pd.DataFrame(history)
history_df


In [ ]:
run_dir = ARTIFACT_ROOT / ("pin_siglip_mlp_" + BACKEND)
run_dir.mkdir(parents=True, exist_ok=True)
torch.save(mlp.state_dict(), run_dir / "model.pt")
pd.DataFrame(valid_x).to_parquet(run_dir / "validation_features.parquet", index=False)
history_df.to_parquet(run_dir / "history.parquet", index=False)
final_probs = np.vstack([torch.softmax(mlp(torch.tensor(valid_x, dtype=torch.float32, device=device)), dim=-1).detach().cpu().numpy()])
final_metrics = compute_multiclass_metrics(valid_df["label_name"], final_probs, classes)
(run_dir / "metrics.json").write_text(json.dumps(final_metrics, ensure_ascii=False, indent=2), encoding="utf-8")
final_metrics


In [ ]:
lora_cfg = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj"],
    use_dora=False,
)
dora_cfg = LoraConfig(
    task_type=TaskType.FEATURE_EXTRACTION,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj"],
    use_dora=True,
)

lora_model = get_peft_model(AutoModel.from_pretrained(MODEL_ID), lora_cfg)
dora_model = get_peft_model(AutoModel.from_pretrained(MODEL_ID), dora_cfg)
lora_model.save_pretrained(ARTIFACT_ROOT / ("pin_siglip_lora_" + BACKEND))
dora_model.save_pretrained(ARTIFACT_ROOT / ("pin_siglip_dora_" + BACKEND))
